In [4]:

import smtplib
import imaplib
import email
from email.message import EmailMessage
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart
from email.mime.image import MIMEImage
from email.utils import make_msgid
from email.header import decode_header
from email.utils import parsedate_to_datetime
from imap_tools import MailBox, A

import json
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import pandas as pd
import re
from collections import defaultdict

import os
import csv

from itertools import zip_longest
import time
import random
from datetime import datetime
# import pytz
import importlib
# from utils import send_gmail

from selenium import webdriver
from selenium.webdriver.firefox.options import Options
from selenium.webdriver.firefox.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException
from selenium.webdriver.support.ui import Select
from webdriver_manager.firefox import GeckoDriverManager
from selenium.webdriver.firefox.firefox_profile import FirefoxProfile
from urllib.parse import urlparse
from projudi_bot import ProjudiBot
import sys



bot = ProjudiBot()
bot.executar()

# pega sessão pronta
session = bot.session

# OU pega cookies
cookies = bot.exportar_cookies()

def digitar(texto, elemento):
    for letra in texto:
        elemento.send_keys(letra)
        time.sleep(random.randint(1, 6) / 25)

#document.cookie.split('; ').reduce((acc, cookie) => {const [name, value] = cookie.split('=');acc[name] = value;return acc;}, {});
# pafonso - kvgm hvzm bxbz qibz; ouys uorp vqpr fqig
#user pafonso - pafonso.2vsj@gmail.com
# senha_app = 'ptsd daec xeyw dxgv'
# usuario = 'iguedes953@gmail.com'
tempo_espera = random.uniform(1, 5)
senha_app = 'ouysuorpvqprfqig'
usuario = 'pafonso.2vsj@gmail.com'
IMAP_SERVER = 'imap.gmail.com'
NUM_MAX_EMAILS = 100
PADRAO_ASSUNTO = re.compile(r"2.*VSJ", re.IGNORECASE)


# Configurações do SMTP
SMTP_SERVER = 'smtp.gmail.com'
SMTP_PORT = 587
SMTP_USER = usuario
SMTP_PASSWORD = senha_app
REMETENTE = SMTP_USER
DESTINATARIO = 'pafonso-2vsj@tjba.jus.br'

def encaminhar_email_completo(msg):
    email = EmailMessage()

    # Assunto com prefixo "Enc:"
    email['Subject'] = f"Enc: {msg.subject or '(sem assunto)'}"
    email['From'] = REMETENTE
    email['To'] = DESTINATARIO

    # Corpo do e-mail com conteúdo original
    corpo_original = msg.text or msg.html or "(sem conteúdo)"

    corpo = f"""
Encaminhado automaticamente.

────────────────────────────────────
📨 Remetente original: {msg.from_}
📅 Data: {msg.date.strftime('%d/%m/%Y %H:%M')}
📄 Assunto original: {msg.subject}

📎 Anexos: {[anexo.filename for anexo in msg.attachments if anexo.filename]}

────────────────────────────────────

{corpo_original}
"""
    email.set_content(corpo)

    # Anexar todos os PDFs encontrados
    pdfs_encontrados = 0
    for anexo in msg.attachments:
        nome = anexo.filename or ''
        if nome.lower().endswith('.pdf'):
            email.add_attachment(anexo.payload, maintype='application', subtype='pdf', filename=nome)
            pdfs_encontrados += 1

    if pdfs_encontrados == 0:
        print(f"⚠️ Nenhum PDF encontrado em: {msg.subject}")
        return

    # Enviar via SMTP
    try:
        with smtplib.SMTP(SMTP_SERVER, SMTP_PORT) as smtp:
            smtp.starttls()
            smtp.login(SMTP_USER, SMTP_PASSWORD)
            smtp.send_message(email)
        print(f"✅ E-mail '{msg.subject}' encaminhado com sucesso para {DESTINATARIO}")
    except Exception as e:
        print(f"❌ Erro ao encaminhar e-mail: {e}")

# import csv
# import os

dados = []
path_csv = 'protocolo_email_projudi.csv'
msg_ids_registrados = set()

CAMPOS = [
    'processo', 'num_oficio', 'email_destino', 'status',
    'data_envio', 'hora_envio', 'registrar_envio',  'resposta', 'msg_id',
    'url_oficio', 'url_processo', 'url_recebimento', 'url_baixa', 'assunto',
]

# Cria arquivo se não existir
if not os.path.exists(path_csv):
    print(f"[INFO] Arquivo {path_csv} não existe. Criando novo.")
    with open(path_csv, 'w', encoding='utf-8', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=CAMPOS, delimiter=';')
        writer.writeheader()

# Carrega CSV e remove duplicados
chaves_vistas = set()  # chave composta para evitar duplicados
with open(path_csv, encoding='utf-8', newline='') as f:
    reader = csv.DictReader(f, delimiter=';')
    for row in reader:
        # Evita linha de cabeçalho duplicado
        if str(row.get('processo', '')).strip().lower() == 'processo':
            continue

        # Normaliza campos
        for campo in CAMPOS:
            if campo not in row:
                row[campo] = ''
            elif row[campo] is None:
                row[campo] = ''
            else:
                row[campo] = str(row[campo]).strip()

        # Garantir que 'resposta' não fique vazia
        if not row.get('resposta', '').strip():
            row['resposta'] = ''

        # Cria chave única por assunto + data_envio + hora_envio
        chave = (
            (row.get('assunto', '').strip().lower() or '') + '_' +
            (row.get('data_envio', '').strip() or '') + '_' +
            (row.get('hora_envio', '').strip() or '')
        )
        msg_id = row.get('msg_id', '').strip()

        # Ignora duplicados
        if chave in chaves_vistas or (msg_id and msg_id in msg_ids_registrados):
            continue

        chaves_vistas.add(chave)
        if msg_id:
            msg_ids_registrados.add(msg_id)

        dados.append(row)

# Sobrescreve o CSV sem duplicados
with open(path_csv, 'w', encoding='utf-8', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=CAMPOS, delimiter=';')
    writer.writeheader()
    writer.writerows(dados)

print(f"[INFO] CSV atualizado sem duplicados. Total de registros: {len(dados)}")
print(f"🧾 {len(msg_ids_registrados)} mensagens já registradas.")



# document.cookie.split('; ').reduce((acc, cookie) => {const [name, value] = cookie.split('=');acc[name] = value;return acc;}, {});
# Abre a página dos ofícios já logado

link_base = 'https://projudi.tjba.jus.br/projudi/'
oficios = 'listagens/CumprimentoCartorio?tipo=oficio&acao=expedidos'
cumprimento_cartorio = 'listagens/AnalisarCumprimentoCartorio'
analise_movimentacao = 'projudi/cadastros/AnalisarMovimentacao'
url_oficios = link_base + oficios
url_cumprimento_cartorio = link_base + cumprimento_cartorio

# Faz a requisição já logado com requests
url = 'https://projudi.tjba.jus.br/projudi/listagens/CumprimentoCartorio?tipo=oficio'

parsed = urlparse(url)
cookie_domain = parsed.hostname 

print(parsed.hostname)  # ➜ 'projudi.tjba.jus.br'
print(parsed.path)


# cookies = {
#     'JSESSIONID':"8BBA4E8FFB4F1298520AC432CBA85954.tomcat09-01",
#     # 'ADC_CONN_539B3595F4E': "81875C25DA191823F9FF7B7722477FAFC690CBD60DF53CBBC63F5F42BAD5246195272EF2641A1CAF", 
#     # 'ADC_REQ_2E94AF76E7': "FC548B0AB68C9C8B7DC0E04A12A63D9F5460B5F1DE741793F56F3D2FA0BDCB0F218B795FCAA762CA", 
#     # 'ADRUM': "s~1774908668005&r~aHR0cHMlM0ElMkYlMkZwcm9qdWRpLnRqYmEuanVzLmJyJTJGcHJvanVkaSUyRg" 
#     }

# cookies = bot.exportar_cookies()
# session = bot.session()

# Simula um navegador real

# headers = {
#     'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 '
#                   '(KHTML, like Gecko) Chrome/115.0.0.0 Safari/537.36',
#     'Accept-Language': 'pt-BR,pt;q=0.9,en;q=0.8',
#     'Referer': url_oficios,  # onde o clique teria ocorrido
#     'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8',
# }

# response = requests.get(url_oficios, headers=headers, cookies=cookies)
response = session.get(url_oficios)
# Analisa o HTML retornado
html = response.text
soup = BeautifulSoup(html, 'html.parser')
# print(soup.prettify()[:200])
print(url_oficios)

html = response.text
# 3. Faz o parser da página com os links
soup = BeautifulSoup(html, 'html.parser')
# 🔹 2. Encontra o <a> com o texto "última"
# link_ultima = soup.find('a', string=re.compile(r'ultima', re.IGNORECASE))

# 2. Encontra todos os links do tipo goToPage(N)
ultima_pagina = '99'
# links_paginas = soup.find_all('a', href=re.compile(r'goToPage\(\d+\)'))
# print(links_paginas[-1])
# if links_paginas[-1]:
#     match = re.search(r'goToPage\((\d+)\)', links_paginas[-1]['href'])
#     if match:
#         ultima_pagina = match.group(1)      # ➜ '89' (string)
#         # numero_int = int(numero_str)     # ➜ 89 (inteiro)

#         print("Como string:", ultima_pagina)
        # print("Como número:", numero_int)

# Simula o "clique" na última página
data_oficio = {
    'tipo': 'oficio',
    'acao': 'expedidos',
    'codTipoJustica': "2",
    'pagina': ultima_pagina,# <-- Este número veio de goToPage(89)
    'coluna' :'CumprimentoCartorio.CODCUMPRIMENTO',
    'ordem':"ASC"
}



# 1. Requisição como se fosse o clique no link
# response = requests.post(url_oficios, headers=headers, cookies=cookies, data=data_oficio)
# POST usando a mesma sessão
response = session.post(url_oficios, data=data_oficio)
response_analisar_cumprimento_cart = session.post(url_cumprimento_cartorio)
# response_analisar_cumprimento_cart = requests.post(url_cumprimento_cartorio, headers=headers, cookies=cookies)
print(response.url)        # Verifica se foi redirecionado
print(response.status_code)
print(response.text[:500])  # Mostra o começo do HTML
# response = requests.get(url_oficios, headers=headers, cookies=cookies)
# print(response.status_code)
# print(response.text)  # ou .content para binário

html = response.text
# 3. Faz o parser da página com os links
soup = BeautifulSoup(html, 'html.parser')
link_ultima_pagina = soup.find('a', href=re.compile(r'goToPage\(\d+\)'))
print(link_ultima_pagina)
# limpo = limpar_html_para_email(html, 'html.parser')


def send_gmail(txt_oficio, processo, numero_oficio, 
               url_oficio, url_processo, url_recebimento, url_baixa,
               arquivo_csv, email_destino): 

    # Usar o arquivo CSV recebido por parâmetro (garanta que PATH_CSV = arquivo_csv)
    path_csv = 'protocolo_email_projudi.csv'

    # # Campos do CSV - nomes consistentes e sem acentos
    # CAMPOS = [
    #     'processo', 'num_oficio', 'email_destino', 'status',
    #     'data_envio', 'hora_envio', 'registrar_envio', 'msg_id',
    #     'url_oficio', 'url_processo', 'url_recebimento', 'url_baixa', 'assunto',
    # ]

    # if not os.path.exists(path_csv):
    #     print(f"[INFO] Arquivo {path_csv} não existe. Criando novo.")
    #     with open(path_csv, 'w', encoding='utf-8', newline='') as f:
    #         writer = csv.DictWriter(f, fieldnames=CAMPOS, delimiter=';')
    #         writer.writeheader()

    # Ler o CSV
    with open(path_csv, 'r', encoding='utf-8', newline='') as f:
        reader = csv.DictReader(f, delimiter=';')
        dados = list(reader)
    print(f"[INFO] Linhas carregadas do CSV: {len(dados)}")
   

    # Se CSV só tiver cabeçalho, criar uma linha para envio
    if len(dados) == 0:
        print("[INFO] CSV vazio, criando nova linha para envio.")
        dados.append({
            'processo': processo,
            'num_oficio': '',
            'email_destino': email_destino,
            'status': '',
            'data_envio': '',
            'hora_envio': '',
            'registrar_envio': '',
            'msg_id': '',
            'url_oficio': '',
            'url_processo': '',
            'url_recebimento': '',
            'url_baixa': '',
            'assunto': '',
        })
    linha_existente = next((
        linha for linha in dados
        if linha.get('processo', '').strip() == processo
        and linha.get('num_oficio', '').strip() == numero_oficio
        ), None)
    if linha_existente:
        status = linha_existente.get('status', '').strip().lower()
        juntado = linha_existente.get('registrar_envio', '').strip().lower() in ('juntado', 'cumprido', 'baixado')
        if status == 'enviado' and juntado:
            print(f"""[INFO] Ofício {numero_oficio} 
                  já foi enviado e juntado, cumprido ou baixado. Nada a fazer.""")
            return "enviado_e_juntado/cumprido/baixado"
        
        elif status == 'enviado' and not juntado:
            print(f"[INFO] Ofício {numero_oficio} já foi enviado. Realizando apenas juntada.")
            remetente = 'pafonso.2vsj@gmail.com'
            data_envio = linha_existente['data_envio']
            numero_oficio = linha_existente['num_oficio']
            status = linha_existente.get('status', '').strip().lower()
            hora_envio = linha_existente['hora_envio']
            destinatario = linha_existente['email_destino']
            print("[INFO] Email enviado, mas juntada pendente. Iniciando juntada com Selenium...")
            profile = FirefoxProfile()
            # Define o User-Agent (exatamente o mesmo que você usou no requests)
            profile.set_preference("general.useragent.override",
                "Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:115.0) Gecko/20100101 Firefox/115.0"
            )
            # Define o idioma do navegador (opcional)
            profile.set_preference("intl.accept_languages", "pt-BR,pt;q=0.9,en;q=0.8")
            # Cria opções e adiciona o perfil
            options = Options()
            options.profile = profile
            # Inicializa o driver com GeckoDriverManager
            driver = webdriver.Firefox(
                service=Service(GeckoDriverManager().install()),
                options=options
            )
            # Define o tamanho da janela (parece mais com uso humano)
            driver.set_window_size(1280, 800)
            # Abre a página desejada
            driver.get(link_base)
            for name, value in cookies.items():
                driver.add_cookie({
                    'name': name,
                    'value': value,
                    'path': '/',           # geralmente '/'
                    'domain': cookie_domain,  # importante: deve bater com o domínio acessado
                    'secure': True         # define se o cookie é só para HTTPS
                    })
            for cookie in driver.get_cookies():
                print(cookie)
            try:
                # Exemplo de uso do driver Selenium (garanta que driver, tempo_espera e função digitar estão definidos)
                driver.get(url_recebimento)
                wait = WebDriverWait(driver, 20)
                driver.execute_script("window.scrollBy(0, 567);")

                # Preencher código da movimentação
                codigo_movimentacao = wait.until(EC.presence_of_element_located((By.ID, "seqCategoriaMovimentacao")))
                codigo_movimentacao.clear()
                codigo_movimentacao.send_keys("11383")
                time.sleep(tempo_espera)
                # <img id="btnBuscaMovimentacao" style="vertical-align:middle;" border="0" src="/projudi/imagens/botoes/bot-busca.gif" title="Botão buscar Movimentacao">

                # Aguarda o botão estar visível e clica
                wait = WebDriverWait(driver, 5)
                botao = wait.until(EC.element_to_be_clickable((By.ID, "btnBuscaMovimentacao")))
                botao.click()
                time.sleep(tempo_espera)

                # Preencher campo observação
                campo_observacao = wait.until(EC.presence_of_element_located((By.ID, "observacao")))
                observacao = f"Enviado por email o Ofício- {numero_oficio} em {data_envio} às {hora_envio} hs, para: {destinatario}, link: {url_oficio}"
                campo_observacao.clear()
                digitar(observacao, campo_observacao)

                driver.execute_script("window.scrollBy(0, 567);")
                # Clicar para concluir
                botao_concluir = wait.until(EC.element_to_be_clickable((By.ID, "Concluir")))
                botao_concluir.click()
                time.sleep(tempo_espera)

                # Espera o alerta aparecer (até 10 segundos, ajustável)
                alert = WebDriverWait(driver, 10).until(EC.alert_is_present())

                # Exibe e aceita o alerta
                print(f"Alerta exibido: {alert.text}")
                alert.accept()

                time.sleep(3)
                print("✅ Juntada realizada com sucesso!")

                # Atualizar status de juntada
                linha_existente['registrar_envio'] = 'juntado'
                with open(path_csv, 'w', encoding='utf-8', newline='') as f:
                    writer = csv.DictWriter(f, fieldnames=CAMPOS, delimiter=';')
                    writer.writeheader()
                    writer.writerows(dados)
                print(f"[✓] Dados de juntada atualizados no CSV.")
                driver.close()
                time.sleep(3)
                return f'juntada realizada no projudi referente ao {linha_existente["numero_oficio"]}'
                

            except Exception as e:
                    print(f"❌ Não foi possível realizar a juntada: {e}")
                    driver.close()
####################################################################################
                    
    else:
        print(f"[INFO] Enviando email para processo: {processo} e ofício: {numero_oficio}")
        remetente = 'pafonso.2vsj@gmail.com'  # Seu email
        senha = senha_app  # Sua senha de app (defina globalmente)
        # destinatario = 'igusilva@tjba.jus.br'
        destinatario = email_destino # Recebedor do email
        assunto = f'2ªVSJ - {numero_oficio} Referente ao Proc Nº: {processo}'
        logo_cid = make_msgid(domain="tjba.jus.br")[1:-1]
        html = f"""
        <html>
        <head></head>
        <body style="font-family: Arial, sans-serif; font-size: 14px;">
            <p>Prezado(a) Senhor(a),</p>
            <p>Este endereço de email é <b>apenas para fins de envio AUTOMÁTICO de Ofícios</b>. Encaminhamos o documento abaixo.</p>
            <p>Por favor, acuse o recebimento.
            <P>Caso seja necessário enviar um Ofício de resposta, envie-o para o email: <b>pafonso-2vsj@tjba.jus.br</b></p>
            <br>
            <p style="margin-bottom: 10px;">Atenciosamente,<br><br>
            2ª Vara do Sistema dos Juizados Especiais – TJBA</p>
            <br><br>
            <!-- Brasão centralizado -->
            <div style="text-align: center;" align="center">
                <img src="cid:{logo_cid}" style="height:80px;" alt="Brasão TJBA">
            </div>
            
            <!-- Cabeçalho institucional centralizado -->

            <div style="margin-top: 2px; text-align: center;">
    
            2ª Vara dos Juizados Especiais de Paulo Afonso – BA<br>
            Rua das Caraibeiras, 420, Quadra 04 1º Andar, General Dutra - Fórum de Paulo Afonso <br>
            pafonso-2vsj@tjba.jus.br // Tel.: (75) 32818 - 8372
            </div> <br>
            <hr style="margin: 20px auto; width: 60%; border: none; border-top: 1px solid #ccc;">
            <div style="text-align: justify; margin: 0 auto; width: 80%; max-width: 600px;">
            {txt_oficio}
            </div>
        </body>
        </html> """
        # Montar a mensagem com HTML + imagem
        msg = MIMEMultipart("related")
        msg["Subject"] = assunto
        msg["From"] = remetente
        msg["To"] = destinatario
        msg.add_header('Return-Receipt-To', remetente)
        msg.add_header('Return-Receipt-To', remetente)
        msg_id = make_msgid().strip('<>')
        msg['Message-ID'] = msg_id
        # Parte alternativa com HTML
        alt_part = MIMEMultipart("alternative")
        html_part = MIMEText(html, 'html', 'utf-8')
        alt_part.attach(html_part)
        msg.attach(alt_part)
        # Anexar o brasão embutido
        with open("tjba.png", 'rb') as img_file:
            img = MIMEImage(img_file.read(), _subtype="png")
            img.add_header('Content-ID', f'<{logo_cid}>')
            img.add_header('Content-Disposition', 'inline', filename="brasao_tjba.png")
            msg.attach(img)
            
        try:
            with smtplib.SMTP_SSL("smtp.gmail.com", 465) as servidor:
                servidor.login(remetente, senha)
                servidor.send_message(msg)
            print("✅ E-mail enviado com sucesso.")
            email_enviado = True
            # Atualizar campos da linha
            data_hora = datetime.now()
            data_envio = data_hora.strftime('%d/%m/%Y')
            hora_envio = data_hora.strftime('%H:%M:%S')
            nova_linha = {
                'processo': processo,
                'num_oficio': numero_oficio,
                'email_destino': destinatario,
                'status': 'Enviado',
                'data_envio': data_envio,
                'hora_envio': hora_envio,
                'registrar_envio': '',
                'msg_id': msg_id,
                'url_oficio': url_oficio,
                'url_processo': url_processo,
                'url_recebimento': url_recebimento,
                'url_baixa': url_baixa,
                'assunto': assunto,
            }
            # ✅ Evita duplicar se já existir no CSV
            ja_existe = any(
                linha['processo'].strip() == processo and linha['num_oficio'].strip() == numero_oficio
                for linha in dados
            )
            if not ja_existe:
                dados.append(nova_linha)
                print(f"[✓] Nova linha adicionada para processo {processo} e ofício {numero_oficio}.")
            else:
                print(f"[INFO] Já existe um registro para processo {processo} e ofício {numero_oficio}. Não adicionando duplicado.")
            # Adiciona a nova linha
           
            # Regravar CSV atualizado
            with open(path_csv, 'w', encoding='utf-8', newline='') as f:
                writer = csv.DictWriter(f, fieldnames=CAMPOS, delimiter=';')
                writer.writeheader()
                writer.writerows(dados)
            print(f"[✓] Dados atualizados no CSV.")
            print(f"✅ Ofício {numero_oficio} enviado para {destinatario} - {processo}")
        except Exception as e:
                print(f"❌ Erro ao enviar o e-mail: {e}")
                email_enviado = False

        if email_enviado:
            profile = FirefoxProfile()
            # Define o User-Agent (exatamente o mesmo que você usou no requests)
            profile.set_preference("general.useragent.override",
                "Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:115.0) Gecko/20100101 Firefox/115.0"
            )
            # Define o idioma do navegador (opcional)
            profile.set_preference("intl.accept_languages", "pt-BR,pt;q=0.9,en;q=0.8")
            # Cria opções e adiciona o perfil
            options = Options()
            options.profile = profile
            # Inicializa o driver com GeckoDriverManager
            driver = webdriver.Firefox(
                service=Service(GeckoDriverManager().install()),
                options=options
            )
            # Define o tamanho da janela (parece mais com uso humano)
            driver.set_window_size(1280, 800)
            # Abre a página desejada
            driver.get(link_base)
            for name, value in cookies.items():
                driver.add_cookie({
                        'name': name,
                        'value': value,
                        'path': '/',           # geralmente '/'
                        'domain': cookie_domain,  # importante: deve bater com o domínio acessado
                        'secure': True         # define se o cookie é só para HTTPS
                    })
            for cookie in driver.get_cookies():
                print(cookie)
            try:
                # Exemplo de uso do driver Selenium (garanta que driver, tempo_espera e função digitar estão definidos)
                driver.get(url_recebimento)
                wait = WebDriverWait(driver, 20)
                driver.execute_script("window.scrollBy(0, 567);")
                # Preencher código da movimentação
                codigo_movimentacao = wait.until(EC.presence_of_element_located((By.ID, "seqCategoriaMovimentacao")))
                codigo_movimentacao.clear()
                codigo_movimentacao.send_keys("11383")
                time.sleep(tempo_espera)
                # <img id="btnBuscaMovimentacao" style="vertical-align:middle;" border="0" src="/projudi/imagens/botoes/bot-busca.gif" title="Botão buscar Movimentacao">
                # Aguarda o botão estar visível e clica
                wait = WebDriverWait(driver, 5)
                botao = wait.until(EC.element_to_be_clickable((By.ID, "btnBuscaMovimentacao")))
                botao.click()
                time.sleep(tempo_espera)
                # Preencher campo observação
                campo_observacao = wait.until(EC.presence_of_element_located((By.ID, "observacao")))
                observacao = f"Email enviado por {remetente} - {numero_oficio} em {data_envio} às {hora_envio}, para: {destinatario}, link: {url_oficio}"
                campo_observacao.clear()
                digitar(observacao, campo_observacao)
                driver.execute_script("window.scrollBy(0, 567);")
                # Clicar para concluir
                botao_concluir = wait.until(EC.element_to_be_clickable((By.ID, "Concluir")))
                botao_concluir.click()
                time.sleep(tempo_espera)
                # Espera o alerta aparecer (até 10 segundos, ajustável)
                alert = WebDriverWait(driver, 10).until(EC.alert_is_present())
                # Exibe e aceita o alerta
                print(f"Alerta exibido: {alert.text}")
                alert.accept()
                time.sleep(3)
                print("✅ Juntada realizada com sucesso!")
                # Atualizar status de juntada
                nova_linha['registrar_envio'] = 'juntado'
                with open(path_csv, 'w', encoding='utf-8', newline='') as f:
                    writer = csv.DictWriter(f, fieldnames=CAMPOS, delimiter=';')
                    writer.writeheader()
                    writer.writerows(dados)
                print(f"[✓] Dados de juntada atualizados no CSV.")
                driver.close()
                time.sleep(3)
                return f'enviado e juntado {numero_oficio} para o {destinatario}.'
            except Exception as e:
                print(f"❌ Não foi possível realizar a juntada: {e}")
                driver.quit()
                return email_enviado
                # finally:
                #     driver.quit()
todos_links = [
    a for a in soup.find_all('a', href=True)
    if 'DadosProcesso?numeroProcesso=' in a['href']
]
# print(soup)

links_oficios = []
links_processos = []
links_recebimento = []
links_baixa = []

for a in soup.find_all('a', href= True):
    href = a['href'].replace('&amp', '&')
    if href.startswith('/projudi/acoes/VerCumprimento'):
        links_oficios.append(urljoin(link_base, href))
    elif href.startswith('/projudi/listagens/DadosProcesso'):
        links_processos.append(urljoin(link_base, href))
    elif href.startswith('/projudi/movimentacao/MovimentarProcesso?'):
            # href_limpo = href.split('&')[0] #- transforma a pagina de recebimento em movimentação
            url = urljoin(link_base, href)
            links_recebimento.append(url)
    elif href.startswith('/projudi/movimentacao/MarcaRecebimento?tipo=cumprimento&&codCumprimento'):
            links_baixa.append(urljoin(link_base, href))

        
print(len(links_oficios), len(links_processos))


for n_processo, url_processo, url_oficio, url_juntada, url_baixa in zip_longest(
    todos_links, links_processos, links_oficios, links_recebimento, links_baixa):
    tempo_espera = random.uniform(1, 5)  # espera aleatória entre 1 e 5 segundos
    print(f"⏳ Aguardando {tempo_espera:.2f} segundos para próxima requisição...")
    time.sleep(tempo_espera)
    processo = n_processo.get_text(strip=True) if n_processo else 'Processo não identificado'
    
    try:
        response_oficio = session.get(url_oficio)#, headers=headers, cookies=cookies)
        if response_oficio.status_code != 200:
            print(f"❌ Falha ao acessar ofício do processo {processo}")
            continue
        
        sopa_oficio = BeautifulSoup(response_oficio.text, 'html.parser')
        
        # Encontra a div com o texto do ofício
        # div_oficio = sopa_oficio.find('div', attrs={
        #     "style": re.compile(r"width:\s*650px.*font-family:\s*Tahoma", re.IGNORECASE)
        # })
        
        div_oficio = sopa_oficio.find('div', attrs={
                "style": re.compile(r"width:\s*650px.*font-family:\s*Tahoma", re.IGNORECASE)
            })
        if not div_oficio:
            div_oficio = sopa_oficio.find('div', style=lambda s: s and 'width: 650px' in s)

        if not div_oficio:
            div_oficio = sopa_oficio.find('div', attrs={
            'style': 'width: 650px; margin: 0pt auto; font-size: 12px; font-family: Tahoma,Geneva,sans-serif;'
            })

    
        if not div_oficio:
            print(f"❌ Ofício do processo {processo} não encontrado.")
            continue

        
        
        # Transforma o html em texto
        oficio_busca = div_oficio.get_text(separator='\n', strip=True)

        #Envia o email no formato HTML
        oficio_texto = str(div_oficio)#.get_text(separator='\n', strip=True)
        
        
        # Extrai número do ofício usando regex no texto capturado
        match_oficio = re.search(
            r"Of[ií]cio\s*(n[ºo°.:]*)?\s*[:\-]?\s*([\d]{1,4}/[\d]{4}(?:\s*-\s*[A-Z]+)?)",
            oficio_busca,
            re.IGNORECASE
        )
        numero_oficio = match_oficio.group(2).strip() if match_oficio else ""
        
        # Extrai todos os e-mails do texto do ofício
        emails = re.findall(r"[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+", oficio_busca)
        
        # Filtra e-mails removendo remetentes internos
        destinatarios_validos = [
            e for e in emails 
            if not re.search(r"@tjba|@tjbacote|pafonso-2vsj@tjba\.jus\.br", e, re.IGNORECASE)
        ]
        
        email_destino = destinatarios_validos[0] if destinatarios_validos else "Email não encontrado"
        
        # Imprime dados extraídos
        print(f"\n--- Dados Extraídos ---")
        print(f"Processo: {processo}")
        print(f"Número do Ofício: {numero_oficio}")
        print(f"E-mail destinatário: {email_destino}")
        print(f"URL do processo: {url_processo}")
        print(f"URL do ofício: {url_oficio}")
        print("------------FIM DO OFÍCIO------------\n")
        email = send_gmail(txt_oficio = oficio_texto,
                           processo = processo, 
                           url_oficio = url_oficio,
                           url_processo = url_processo, 
                           numero_oficio = numero_oficio,
                           url_recebimento = url_juntada,
                           url_baixa = url_baixa,
                           arquivo_csv = 'path_csv',
                           email_destino = email_destino,)
        # print(oficio_texto)
    
    except Exception as e:
        print(f"❌ Erro ao processar ofício do processo {processo}: {e}")

# html, processo, numero_oficio, 
#                url_oficio, url_processo, url_recebimento, url_baixa,
#                arquivo_csv, email_destino


 Possível sessão inválida
[INFO] CSV atualizado sem duplicados. Total de registros: 60
🧾 59 mensagens já registradas.
projudi.tjba.jus.br
/projudi/listagens/CumprimentoCartorio
https://projudi.tjba.jus.br/projudi/listagens/CumprimentoCartorio?tipo=oficio&acao=expedidos
https://projudi.tjba.jus.br/projudi/listagens/CumprimentoCartorio?tipo=oficio&acao=expedidos
200











<script src="/projudi/dwr/engine.js"></script>
<script src="/projudi/dwr/util.js"></script>


<script type="text/javascript" src="/projudi/scripts/subModal/subModal.js"></script> 
<script type="text/javascript" src="/projudi/dwr/interface/Pesquisa.js"></script>
<script type="text/javascript" src="/projudi/scripts/tabelasProcessuais/tabelasProcessuais.js"></script>


<html>
    <head>
        <meta http-equiv="Content-Type" content="text/html; charset=iso-8859-1">
      
<a alt="primeira" href="javascript:goToPage(1)" title="primeira página"><img border="0" src="/projudi/imagens/seta2_esquerda.gif"/></a>
9 9
⏳ Aguar

In [ ]:
# import websockets
# print(websockets.__version__)